In [29]:
# %pip install numpy
# %pip install pandas
# %pip install pymongo
# %pip install scikit-learn

In [30]:
from itertools import product
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

def get_data_from_mongodb(uri = 'mongodb://localhost:27017', db_name = 'iot_project', collection = 'measurements'):
    try:
        client = MongoClient(uri, serverSelectionTimeoutMS=1000)
        client.admin.command("ping")
        print("Connection successful")

        db = client[db_name]
        collection = db[collection]

        cursor = collection.find({})
        df = pd.DataFrame(list(cursor))
        print(list(df.columns))
        return df
    except ConnectionFailure:
        print("Connection failed")

def clean_data(df):
    #delete high correlation and irrelevant columns
    df = df.drop(columns=['_id', 'node_id', 'raw_temperature', 'raw_humidity', 'raw_line', 'count'])

    #delete rows with more than 90% missing data
    df = df.dropna(thresh=int(0.9 * len(df.columns)), axis=0)

    #delete duplicate rows
    df = df.drop_duplicates()

    #delete statistic outliers
    for column in df.select_dtypes(include=['float64', 'int64']).columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 20 * iqr
        upper_bound = q3 + 20 * iqr
        df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

    #delete realistic outliners
    df.loc[
        (df["temperature"] < -20) |
        (df["temperature"] > 60),
        "temperature"
    ] = np.nan

    df.loc[
        (df["humidity"] < 0) |
        (df["humidity"] > 100),
        "humidity"
    ] = np.nan

    #fill in blanks
    df[['temperature', 'humidity']] = df[['temperature', 'humidity']].interpolate()

    #set time values to timestamp and make sure they are sorted
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.sort_values("timestamp").reset_index(drop=True)

    return df

def get_feature_columns(df):
    return [col for col in df.columns
            if col.startswith("temp_lag_")
            or col.startswith("hum_lag_")
            or col.startswith("temp_roll_")
            or col.startswith("hum_roll_")]

def prepare_features(df, lags, windows):


    for lag in lags:
        df[f'temp_lag_{lag}'] = df['temperature'].shift(lag)
        df[f'hum_lag_{lag}'] = df['humidity'].shift(lag)

    #average of the previous w measurements, excluding the current measurement
    for w in windows:
        df[f"temp_roll_mean_{w}"] = df["temperature"].shift(1).rolling(w).mean()
        df[f"hum_roll_mean_{w}"] = df["humidity"].shift(1).rolling(w).mean()

    #prepare the last row for predictions
    df["target_temp"] = df["temperature"].shift(-1)
    df["target_hum"] = df["humidity"].shift(-1)

    return df

def get_splits(df):
    feature_cols = get_feature_columns(df)
    #get all rows except the last one so that all data exists for training
    model_df = df.dropna(subset=feature_cols + ["target_temp", "target_hum"])
    x = model_df[feature_cols]
    y = model_df[["target_temp", "target_hum"]]

    train_end = int(0.8 * len(model_df))
    val_end = int(0.9 * len(model_df))

    x_train = x.iloc[:train_end]
    y_train = y.iloc[:train_end]

    x_val = x.iloc[train_end:val_end]
    y_val = y.iloc[train_end:val_end]

    x_test = x.iloc[val_end:]
    y_test = y.iloc[val_end:]
    return x_train, y_train, x_val, y_val, x_test, y_test

def create_model(df, #regulariztion can be None, "L1" or "L2"
                     regularization = "L2",
                 alpha=1.0):
    x_train, y_train, x_val, y_val, x_test, y_test = get_splits(df)


    if regularization is None:
        curr_model = make_pipeline(
        StandardScaler(), #sets every value as value-(mean of col)
        LinearRegression(fit_intercept=True))
    elif regularization == "L2":
        curr_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha, fit_intercept=True))
    elif regularization == "L1":
        curr_model = make_pipeline(
        StandardScaler(),
        Lasso(alpha=alpha, fit_intercept=True))
    else:
        raise ValueError("Invalid regularization type. Choose None, 'L1', or 'L2'.")

    curr_model.fit(x_train, y_train)

    val_pred = curr_model.predict(x_val)
    val_mse = mean_squared_error(y_val, val_pred)
    val_r2 = r2_score(y_val, val_pred)

    return {
        "model": curr_model,
        "val_MSE": val_mse,
        "val_R^2": val_r2,
    }

def find_best_model(curr_df):
    best_mse = None
    best_params = None
    best_model = None
    best_df = None

    #lags are delay of 20s, 1m, 3m, 6m, 15m, 30m, 1h, 3h, 6h, 12h, 24h
    lags = [1, 3, 9, 18, 45, 90, 180, 540, 1080, 2160, 4320]
    all_lags = [lags[:l] for l in range(1, len(lags) + 1)]
    #windows of 1m, 3m, 15m, 1h
    windows = [3, 9, 45, 180]
    all_windows = [windows[:w] for w in range(1, len(windows) + 1)]
    regularization_types = [None, "L1", "L2"]
    alpha_values = [0.05, 0.1, 0.15, 0.2]
    for lags, windows, reg_type, a in product(
        all_lags, all_windows, regularization_types, alpha_values
    ):
        # print("checking lags: ", lags, "windows: ", windows, "reg_type: ", reg_type, "alpha: ", a)
        if reg_type is None and a != 0.05:
            continue
        prediction_df = prepare_features(curr_df.copy(), lags, windows)
        res = create_model(prediction_df, reg_type, a)
        curr_model = res["model"]
        curr_mse = res['val_MSE']
        if best_mse is None or curr_mse < best_mse:
            best_mse = curr_mse
            best_model = curr_model
            best_df = prediction_df
            best_params = {
                "lags": lags,
                "window": windows,
                "regularization": reg_type,
                "alpha": a
            }
            print(f"New best model found with MSE: {best_mse} and params: {best_params}")
    return best_model, best_params, best_mse, best_df


In [31]:
raw_data = get_data_from_mongodb()

display(raw_data)
display(raw_data.describe())

Connection successful
['_id', 'timestamp', 'node_id', 'temperature', 'humidity', 'count', 'raw_temperature', 'raw_humidity', 'raw_line']


,_id,timestamp,node_id,temperature,humidity,count,raw_temperature,raw_humidity,raw_line
0,6a0b41f94d23a6034fbe0721,2026-05-18 16:44:41.444,1,26.64,45.17,294,6624,1332,"node=1,temp=6624,humidity=1332,count=294"
1,6a0b42224d23a6034fbe0722,2026-05-18 16:45:22.478,1,26.62,45.20,296,6622,1333,"node=1,temp=6622,humidity=1333,count=296"
2,6a0b42374d23a6034fbe0723,2026-05-18 16:45:43.017,1,26.60,45.23,297,6620,1334,"node=1,temp=6620,humidity=1334,count=297"
3,6a0b424c6b194568a8d38e22,2026-05-18 16:46:04.975,1,26.58,45.36,298,6618,1338,"node=1,temp=6618,humidity=1338,count=298"
4,6a0b42606b194568a8d38e23,2026-05-18 16:46:24.053,1,26.56,45.46,299,6616,1341,"node=1,temp=6616,humidity=1341,count=299"
...,...,...,...,...,...,...,...,...,...
15321,6a10d84c001f7d2d8940e2ad,2026-05-22 22:27:24.803,1,26.33,46.29,17765,6593,1367,"node=1,temp=6593,humidity=1367,count=17765"
15322,6a10d861001f7d2d8940e2ae,2026-05-22 22:27:45.321,1,26.34,46.19,17766,6594,1364,"node=1,temp=6594,humidity=1364,count=17766"
15323,6a10d875001f7d2d8940e2af,2026-05-22 22:28:05.857,1,26.37,46.23,17767,6597,1365,"node=1,temp=6597,humidity=1365,count=17767"
15324,6a10d88a001f7d2d8940e2b0,2026-05-22 22:28:26.396,1,26.38,46.23,17768,6598,1365,"node=1,temp=6598,humidity=1365,count=17768"


,timestamp,node_id,temperature,humidity,count,raw_temperature,raw_humidity
count,15326,15326.0,15326.000000,15326.000000,15326.000000,15326.000000,15326.000000
mean,2026-05-21 02:26:34.923432,1.0,26.005726,47.613185,10048.834791,6560.572556,1408.612554
min,2026-05-18 16:44:41.444000,1.0,23.920000,26.810000,0.000000,6352.000000,778.000000
25%,2026-05-20 04:55:48.269750,1.0,25.830000,47.580000,6275.250000,6543.000000,1407.000000
50%,2026-05-21 02:46:47.827000,1.0,25.930000,47.780000,10106.500000,6553.000000,1414.000000
75%,2026-05-22 00:37:47.373250,1.0,26.150000,47.960000,13937.750000,6575.000000,1419.000000
max,2026-05-22 22:28:46.919000,1.0,38.820000,53.010000,17769.000000,7842.000000,1585.000000
std,NaN,0.0,0.390387,0.959368,4535.955613,39.038729,29.883226


In [32]:
data = clean_data(raw_data)
display(data)
display(data.describe())


,timestamp,temperature,humidity
0,2026-05-18 16:44:41.444,26.64,45.17
1,2026-05-18 16:45:22.478,26.62,45.20
2,2026-05-18 16:45:43.017,26.60,45.23
3,2026-05-18 16:46:04.975,26.58,45.36
4,2026-05-18 16:46:24.053,26.56,45.46
...,...,...,...
15321,2026-05-22 22:27:24.803,26.33,46.29
15322,2026-05-22 22:27:45.321,26.34,46.19
15323,2026-05-22 22:28:05.857,26.37,46.23
15324,2026-05-22 22:28:26.396,26.38,46.23


,timestamp,temperature,humidity
count,15310,15310.000000,15310.000000
mean,2026-05-21 02:30:10.345421,25.998240,47.627576
min,2026-05-18 16:44:41.444000,23.920000,40.220000
25%,2026-05-20 04:59:54.645000,25.830000,47.580000
50%,2026-05-21 02:49:32.074500,25.930000,47.780000
75%,2026-05-22 00:39:09.495750,26.150000,47.960000
max,2026-05-22 22:28:46.919000,29.110000,53.010000
std,NaN,0.292947,0.835900


In [33]:
model, params, mse, prediction_data = find_best_model(data)
x_train, y_train, x_val, y_val, x_test, y_test = get_splits(prediction_data)

test_pred = model.predict(x_test)
test_mse = mean_squared_error(y_test, test_pred)
test_r2 = r2_score(y_val, test_pred)
print("Final model test MSE: ", test_mse)
print("Final model test R^2: ", test_r2)

New best model found with MSE: 0.0014925779422909331 and params: {'lags': [1], 'window': [3], 'regularization': None, 'alpha': 0.05}
New best model found with MSE: 0.0014925418412889985 and params: {'lags': [1], 'window': [3], 'regularization': 'L2', 'alpha': 0.05}
New best model found with MSE: 0.0014925061253693822 and params: {'lags': [1], 'window': [3], 'regularization': 'L2', 'alpha': 0.1}
New best model found with MSE: 0.0014924707735976064 and params: {'lags': [1], 'window': [3], 'regularization': 'L2', 'alpha': 0.15}
New best model found with MSE: 0.0014924357659331679 and params: {'lags': [1], 'window': [3], 'regularization': 'L2', 'alpha': 0.2}
New best model found with MSE: 0.0014914199716662133 and params: {'lags': [1, 3], 'window': [3], 'regularization': None, 'alpha': 0.05}
New best model found with MSE: 0.0014912815375594574 and params: {'lags': [1, 3], 'window': [3], 'regularization': 'L2', 'alpha': 0.05}
New best model found with MSE: 0.0014911575227852573 and params: 

In [34]:
instants_to_predict = 1000
predictions = []
prediction_data = prepare_features(
    data.copy(),
    params["lags"],
    params["window"]
).reset_index(drop=True)


for i in range(instants_to_predict):
    latest_row = prediction_data[get_feature_columns(prediction_data)].iloc[[-1]]
    curr_pred = model.predict(latest_row) #[[temp, hum]]
    predictions.append(curr_pred[0])
    prediction_data.loc[len(prediction_data), ["temperature", "humidity"]] = curr_pred[0]


    prediction_data = prepare_features(
    prediction_data,
    params["lags"],
    params["window"]
    ).reset_index(drop=True)

next_10_predictions = predictions[:10]
pred_1h_later = predictions[180]
pred_2h_later = predictions[360]
pred_5h_later = predictions[900]
print("Next 10 predictions: ", next_10_predictions)
print("Prediction 1h later: ", pred_1h_later)
print("Prediction 2h later: ", pred_2h_later)
print("Prediction 5h later: ", pred_5h_later)

Next 10 predictions:  [array([26.3651966 , 46.21244354]), array([26.36891461, 46.18836475]), array([26.36426739, 46.20551389]), array([26.36512263, 46.19628176]), array([26.36341599, 46.20897777]), array([26.36474739, 46.19919937]), array([26.36356644, 46.20962837]), array([26.36456486, 46.20617675]), array([26.363312  , 46.20588515]), array([26.36307382, 46.21019185])]
Prediction 1h later:  [26.35045485 46.34217053]
Prediction 2h later:  [26.35028595 46.49855111]
Prediction 5h later:  [26.32879865 46.75556469]
